# Model Training and Evaluation
This notebook demonstrates training and comparing multiple ML models for RUL prediction.

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data_ingestion import PumpDataIngestion
from feature_engineering import PumpFeatureEngineering
from rul_prediction import RULPredictor

# Set style with fallback for compatibility
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except:
    plt.style.use('seaborn-darkgrid')
%matplotlib inline

## 1. Load and Prepare Data

In [ ]:
# Load sensor data
ingestion = PumpDataIngestion(data_path='../data/raw')
df = ingestion.load_sensor_data('pump_sensor_data.csv')

print(f"Loaded {len(df)} samples")
df.head()

## 2. Feature Engineering

In [ ]:
# Apply feature engineering
fe = PumpFeatureEngineering()
df_features = fe.engineer_all_features(df)

print(f"Original shape: {df.shape}")
print(f"After feature engineering: {df_features.shape}")
print(f"\nFeature columns: {len(fe.get_feature_columns())}")

## 3. Train Models

In [ ]:
# Initialize predictor and train all models
predictor = RULPredictor(model_path='../models')
metrics = predictor.train_all_models(df_features)

## 4. Compare Model Performance

In [ ]:
# Create metrics comparison dataframe
metrics_df = pd.DataFrame(metrics).T
print("Model Performance Comparison:")
metrics_df

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# RMSE comparison
axes[0].bar(metrics_df.index, metrics_df['rmse'], color='steelblue', alpha=0.7)
axes[0].set_title('RMSE Comparison (Lower is Better)', fontweight='bold')
axes[0].set_ylabel('RMSE (days)')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(True, alpha=0.3)

# MAE comparison
axes[1].bar(metrics_df.index, metrics_df['mae'], color='coral', alpha=0.7)
axes[1].set_title('MAE Comparison (Lower is Better)', fontweight='bold')
axes[1].set_ylabel('MAE (days)')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(True, alpha=0.3)

# R² comparison
axes[2].bar(metrics_df.index, metrics_df['r2'], color='green', alpha=0.7)
axes[2].set_title('R² Score (Higher is Better)', fontweight='bold')
axes[2].set_ylabel('R²')
axes[2].tick_params(axis='x', rotation=45)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Feature Importance Analysis

In [ ]:
# Get feature importance for Random Forest
importance_rf = predictor.get_feature_importance(model_name='random_forest', top_n=20)

print("Top 20 Features (Random Forest):")
print(importance_rf)

In [ ]:
# Visualize feature importance
if not importance_rf.empty:
    plt.figure(figsize=(12, 8))
    plt.barh(importance_rf['feature'], importance_rf['importance'], color='teal', alpha=0.7)
    plt.xlabel('Importance', fontweight='bold')
    plt.ylabel('Feature', fontweight='bold')
    plt.title('Top 20 Most Important Features (Random Forest)', fontweight='bold', fontsize=14)
    plt.gca().invert_yaxis()
    plt.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()

## 6. Prediction Visualization

In [ ]:
# Make predictions with the best model
predictions = predictor.predict(df_features)
df_features['predicted_rul'] = predictions

# Sample for visualization
df_viz = df_features[::10].copy()

plt.figure(figsize=(15, 6))
plt.plot(df_viz['timestamp'], df_viz['rul'], label='Actual RUL', linewidth=2, alpha=0.7)
plt.plot(df_viz['timestamp'], df_viz['predicted_rul'], label='Predicted RUL', 
         linewidth=2, linestyle='--', alpha=0.7)
plt.xlabel('Timestamp', fontweight='bold')
plt.ylabel('RUL (Days)', fontweight='bold')
plt.title('Actual vs Predicted RUL', fontweight='bold', fontsize=14)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Error Analysis

In [ ]:
# Calculate prediction errors
errors = df_features['rul'] - df_features['predicted_rul']

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Error distribution
axes[0].hist(errors, bins=50, edgecolor='black', alpha=0.7, color='coral')
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero Error')
axes[0].set_xlabel('Prediction Error (days)', fontweight='bold')
axes[0].set_ylabel('Frequency', fontweight='bold')
axes[0].set_title('Prediction Error Distribution', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Residual plot
axes[1].scatter(df_features['predicted_rul'], errors, alpha=0.3, s=10)
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Predicted RUL (days)', fontweight='bold')
axes[1].set_ylabel('Residual (days)', fontweight='bold')
axes[1].set_title('Residual Plot', fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Mean Error: {errors.mean():.2f} days")
print(f"Std Error: {errors.std():.2f} days")
print(f"Max Overestimate: {errors.max():.2f} days")
print(f"Max Underestimate: {errors.min():.2f} days")

## 8. Save Models

In [ ]:
# Save trained models
predictor.save_models()
print("Models saved successfully!")

## 9. Summary

Key findings:
1. Multiple ML models were trained and compared
2. Best performing model was selected based on RMSE
3. Feature importance reveals which sensors are most predictive
4. Error analysis shows model reliability and areas for improvement

Next steps:
- Fine-tune hyperparameters for better performance
- Add more domain-specific features
- Deploy models in production via the dashboard
- Monitor model performance over time